In [1]:
#cargar datos con pandas
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/deuda_publica_2025_03.csv')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/deuda_publica_2025_03.csv'

In [ ]:
cluster_features = [
    'tasa_final',
    'interes_sobre_saldo',
    'servicio_sobre_saldo',
    'amortizacion_sobre_disposicion',
    'dias_contrato',
    'saldo_periodo'
]

In [ ]:
df['interes_sobre_saldo'] = (
    df['intereses_periodo'] / df['saldo_periodo']
)

df['servicio_sobre_saldo'] = (
    df['pago_servicio_deuda'] / df['saldo_periodo']
)

df['amortizacion_sobre_disposicion'] = (
    df['amortizaciones_periodo'] / df['disposicion_inicial_credito']
)


In [ ]:
mask_tiie = df['tasa_final'].astype(str).str.contains('TIIE', case=False, na=False)

df_tasa_fija = df[~mask_tiie].copy()


In [ ]:
df_tasa_fija['tasa_final'] = pd.to_numeric(
    df_tasa_fija['tasa_final'],
    errors='coerce'
)

df_tasa_fija.dropna(subset=['tasa_final'], inplace=True)

In [ ]:
df = df_tasa_fija


In [ ]:
df_modelo = df.copy()


In [ ]:
df_modelo = df_modelo[
    (df_modelo['saldo_periodo'] > 0) &
    (df_modelo['disposicion_inicial_credito'] > 0)
]

In [ ]:
df_modelo['interes_sobre_saldo'] = (
    df_modelo['intereses_periodo'] / df_modelo['saldo_periodo']
)

df_modelo['servicio_sobre_saldo'] = (
    df_modelo['pago_servicio_deuda'] / df_modelo['saldo_periodo']
)

df_modelo['amortizacion_sobre_disposicion'] = (
    df_modelo['amortizaciones_periodo'] /
    df_modelo['disposicion_inicial_credito']
)

In [ ]:
import numpy as np

num_cols = df_modelo.select_dtypes(include=[np.number]).columns

df_modelo[num_cols] = df_modelo[num_cols].replace(
    [np.inf, -np.inf],
    np.nan
)


In [ ]:
cluster_features = [
    'tasa_final',
    'interes_sobre_saldo',
    'servicio_sobre_saldo',
    'amortizacion_sobre_disposicion',
    'dias_contrato',
    'saldo_periodo'
]

df_modelo = df_modelo.dropna(subset=cluster_features)

In [ ]:
np.isinf(df_modelo[cluster_features]).sum().sum(), \
df_modelo[cluster_features].isna().sum().sum()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

cluster_features = [
    'tasa_final',
    'interes_sobre_saldo',
    'servicio_sobre_saldo',
    'amortizacion_sobre_disposicion',
    'dias_contrato',
    'saldo_periodo'
]

x = df_modelo[cluster_features]
x_scaled = StandardScaler().fit_transform(x)

inertia = []
k_range = range(1, 9)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    km.fit(x_scaled)
    inertia.append(km.inertia_)

plt.figure(figsize=(7, 5))
plt.plot(k_range, inertia, marker='o')
plt.xlabel("numero de clusters (k)")
plt.ylabel("inercia")
plt.title("metodo del codo (inercia k-means)")
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

x_cluster = df_modelo[cluster_features]

scaler = StandardScaler()
x_cluster_scaled = scaler.fit_transform(x_cluster)

kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=20
)

df_modelo['cluster'] = kmeans.fit_predict(x_cluster_scaled)


In [ ]:
df_modelo['cluster'].value_counts()


In [ ]:
df_modelo.groupby('cluster')[cluster_features].mean()


In [ ]:
conteo = (
    df_modelo
    .groupby(['cluster', 'acreedor'])
    .size()
    .reset_index(name='num_creditos')
    .sort_values(['cluster', 'num_creditos'], ascending=[True, False])
)

conteo.head(15)

In [ ]:
proporcion = (
    df_modelo
    .groupby(['cluster', 'acreedor'], as_index=False)
    .size()
)

proporcion['proporcion'] = (
    proporcion
    .groupby('cluster')['size']
    .transform(lambda x: x / x.sum())
)

proporcion = proporcion.sort_values(
    ['cluster', 'proporcion'],
    ascending=[True, False]
)

proporcion.head(15)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

top_acreedores = (
    df_modelo['acreedor']
    .value_counts()
    .head(10)
    .index
)

df_top = df_modelo[df_modelo['acreedor'].isin(top_acreedores)]

tabla = pd.crosstab(
    df_top['cluster'],
    df_top['acreedor'],
    normalize='index'
)

tabla.plot(
    kind='bar',
    stacked=True,
    figsize=(10, 6)
)

plt.ylabel("proporcion dentro del cluster")
plt.xlabel("cluster")
plt.title("distribucion de acreedores por cluster (top 10)")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
sns.boxplot(data=df_modelo, x='cluster', y='tasa_final')
plt.title("tasa final por cluster")
plt.show()

“La comparación de la tasa final por cluster muestra diferencias sistemáticas entre los perfiles identificados, lo que confirma que el modelo separó créditos con distintos costos financieros efectivos.”

In [ ]:
sns.boxplot(data=df_modelo, x='cluster', y='interes_sobre_saldo')
plt.title("interes sobre saldo por cluster")
plt.show()

“El cluster 2 presenta una proporción significativamente mayor de intereses respecto al saldo, lo que evidencia una mayor carga financiera periódica aun con tasas moderadas.”

In [ ]:
sns.boxplot(data=df_modelo, x='cluster', y='servicio_sobre_saldo')
plt.title("servicio de la deuda sobre saldo por cluster")
plt.show()

“El análisis del servicio de deuda muestra que el cluster 2 enfrenta una presión financiera significativamente mayor, al destinar una mayor proporción del saldo al pago periódico.”

In [ ]:
sns.boxplot(data=df_modelo, x='cluster', y='amortizacion_sobre_disposicion')
plt.title("amortizacion sobre disposicion por cluster")
plt.show()

“La amortización relativa distingue claramente perfiles de pago acelerado frente a créditos con amortización mínima y prolongada.”

In [ ]:
sns.scatterplot(
    data=df_modelo,
    x='dias_contrato',
    y='saldo_periodo',
    hue='cluster',
    alpha=0.7
)
plt.title("plazo vs saldo por cluster")
plt.show()


In [ ]:
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt

cluster_features = [
    'tasa_final',
    'interes_sobre_saldo',
    'servicio_sobre_saldo',
    'amortizacion_sobre_disposicion',
    'dias_contrato',
    'saldo_periodo'
]

# Escalado
x = df_modelo[cluster_features]
x_scaled = StandardScaler().fit_transform(x)

# t-SNE
tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate=200,
    random_state=42
)

tsne_result = tsne.fit_transform(x_scaled)

# DataFrame para graficar
df_tsne = df_modelo.copy()
df_tsne['tsne_1'] = tsne_result[:, 0]
df_tsne['tsne_2'] = tsne_result[:, 1]

# Gráfica
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df_tsne,
    x='tsne_1',
    y='tsne_2',
    hue='cluster',
    palette='tab10',
    alpha=0.7
)

plt.title("proyeccion tsne de perfiles financieros")
plt.xlabel("tsne componente 1")
plt.ylabel("tsne componente 2")
plt.legend(title="cluster")
plt.show()

El análisis de clustering aplicado a la deuda pública de la Ciudad de México permitió identificar tres perfiles financieros claramente diferenciados, a partir de variables clave relacionadas con el costo, la carga financiera, la estructura de pagos y el horizonte de los créditos. La segmentación se realizó mediante un enfoque no supervisado, garantizando que los grupos emergieran de los propios datos sin imponer categorías predefinidas.
La selección de tres clusters se encuentra sólidamente justificada tanto desde un punto de vista cuantitativo como interpretativo. El método del codo y el índice de Jambu muestran que la reducción de la inercia presenta rendimientos decrecientes a partir de este número de clusters, lo que indica que subdivisiones adicionales no aportan mejoras sustantivas en la cohesión intracluster. Asimismo, las visualizaciones multidimensionales mediante t‑SNE confirman la existencia de tres grupos bien definidos, sin evidencia de una separación natural adicional que justifique un mayor número de clusters.
Los perfiles financieros identificados pueden resumirse de la siguiente manera:


Cluster 0 – Créditos de comportamiento estable:
Agrupa créditos con tasas finales moderadas, baja carga de intereses y niveles controlados de servicio de la deuda. Este perfil se asocia con condiciones financieras equilibradas y un comportamiento predecible, representando la porción más estable del portafolio.


Cluster 1 – Créditos estructurales de gran volumen y largo plazo:
Se caracteriza por concentrar los mayores plazos y saldos promedio, junto con una amortización reducida del capital. El principal riesgo de este perfil no proviene de pagos elevados en el corto plazo, sino de la acumulación de compromisos financieros a lo largo del tiempo, lo que lo convierte en un riesgo estructural.


Cluster 2 – Créditos de alta intensidad de pago:
Este grupo presenta una elevada proporción de intereses y servicio de la deuda respecto al saldo, así como una amortización acelerada. Aunque las tasas finales no son las más altas, la presión financiera periódica es significativamente mayor, lo que implica mayores exigencias de liquidez.


En conjunto, el análisis muestra que el riesgo financiero de la deuda pública no depende exclusivamente de la tasa de interés, sino de la interacción entre plazo, saldo, estructura del servicio de la deuda y ritmo de amortización. La segmentación en tres clusters permite capturar estas diferencias de forma clara, interpretable y útil para el análisis financiero.

El análisis de clustering identifica tres formas distintas de vivir la deuda pública.
Un primer grupo concentra créditos estables, con pagos manejables y bajo nivel de presión financiera.
Un segundo grupo agrupa deudas de gran volumen y largo plazo, cuyo riesgo se concentra en el futuro.
El tercer grupo incluye créditos que se pagan rápido, pero exigen fuertes salidas de dinero en cada periodo.
Esta segmentación permite entender no solo cuánto se debe, sino dónde y cuándo está el verdadero riesgo, apoyando una mejor toma de decisiones financieras.

In [ ]:
df_modelo.groupby('cluster')[
    ['interes_sobre_saldo', 'servicio_sobre_saldo', 'amortizacion_sobre_disposicion']
].mean()
